# Step 11c — Joint DDQN + SARSA: Tier 2 + Discharge Action (GPU)

Trains the joint two-phase RL model on Google Colab GPU.

**Two-phase MDP:**
- Phase 0 (in-ICU blocs): drug combo actions (0-15), SOFA delta reward
- Phase 1 (discharge terminal): discharge destination (0=Home, 1=Home+Services, 2=Institutional), ±15 readmission reward

**Cross-phase Bellman:** the drug Q-network bootstraps from the discharge Q-network at the last in-ICU bloc, so drug decisions are shaped by expected discharge outcomes.

**Before running:**
1. Upload `rl_dataset_tier2_discharge.parquet` to Google Drive at `MyDrive/CareAI/data/`
2. Set Runtime → Change runtime type → T4 GPU
3. Run all cells top to bottom

**Outputs saved to:** `MyDrive/CareAI/models/tier2_discharge/`

In [ ]:
# ── Cell 1: Mount Drive + check GPU ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found -- running on CPU')

In [ ]:
# ── Cell 2: Install missing packages ─────────────────────────────────────────
!pip install pyarrow --quiet

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────
import os

DRIVE_ROOT  = '/content/drive/MyDrive/CareAI'
DATA_PATH   = os.path.join(DRIVE_ROOT, 'data', 'rl_dataset_tier2_discharge.parquet')
MODEL_DIR   = os.path.join(DRIVE_ROOT, 'models', 'tier2_discharge')
DDQN_DIR    = os.path.join(MODEL_DIR, 'ddqn')
SARSA_DIR   = os.path.join(MODEL_DIR, 'sarsa_phys')

os.makedirs(DDQN_DIR,  exist_ok=True)
os.makedirs(SARSA_DIR, exist_ok=True)

N_DRUG_ACTIONS       = 16
N_DISCHARGE_ACTIONS  = 3
HIDDEN               = 128
LR                   = 1e-4
GAMMA                = 0.99
TAU                  = 0.001
BATCH_SIZE           = 512
DQN_STEPS            = 100_000
SARSA_STEPS          = 80_000
REWARD_THRESHOLD     = 20
REG_LAMBDA           = 5.0
PER_ALPHA            = 0.6
PER_EPSILON          = 0.01
BETA_START           = 0.9
CHECKPOINT_EVERY     = 10_000
LOG_EVERY            = 2_000

DISCHARGE_LABELS = {0: 'Home', 1: 'Home+Services', 2: 'Institutional'}

print('Config OK')
print(f'  Data:      {DATA_PATH}')
print(f'  Model dir: {MODEL_DIR}')
print(f'  Device:    {device}')
print(f'  Batch:     {BATCH_SIZE}')

In [ ]:
# ── Cell 4: Network architecture ─────────────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F

class DuelingDQN(nn.Module):
    def __init__(self, n_input, n_actions, hidden=128, leaky_slope=0.01):
        super().__init__()
        self.fc1     = nn.Linear(n_input, hidden)
        self.bn1     = nn.BatchNorm1d(hidden)
        self.fc2     = nn.Linear(hidden, hidden)
        self.bn2     = nn.BatchNorm1d(hidden)
        self.adv_fc  = nn.Linear(hidden, 64)
        self.adv_out = nn.Linear(64, n_actions)
        self.val_fc  = nn.Linear(hidden, 64)
        self.val_out = nn.Linear(64, 1)
        self.leaky_slope = leaky_slope

    def forward(self, x):
        x   = F.leaky_relu(self.bn1(self.fc1(x)), self.leaky_slope)
        x   = F.leaky_relu(self.bn2(self.fc2(x)), self.leaky_slope)
        adv = F.leaky_relu(self.adv_fc(x), self.leaky_slope)
        adv = self.adv_out(adv)
        val = F.leaky_relu(self.val_fc(x), self.leaky_slope)
        val = self.val_out(val)
        return val + adv - adv.mean(dim=1, keepdim=True)

print('DuelingDQN defined')

In [ ]:
# ── Cell 5: Load data ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import pickle
import time
import torch.optim as optim

print(f'Loading {DATA_PATH} ...')
df = pd.read_parquet(DATA_PATH)
print(f'  {len(df):,} rows, {len(df.columns)} cols')
print(f'  Phase 0 (in-stay):   {(df["phase"]==0).sum():,}')
print(f'  Phase 1 (discharge): {(df["phase"]==1).sum():,}')

state_cols      = [c for c in df.columns if c.startswith('s_') and not c.startswith('s_next_')]
next_state_cols = [c.replace('s_', 's_next_', 1) for c in state_cols]
n_state         = len(state_cols)
print(f'  State features ({n_state}): {state_cols}')

df_train = df[df['split'] == 'train'].reset_index(drop=True)
df_val   = df[df['split'] == 'val'].reset_index(drop=True)
df_test  = df[df['split'] == 'test'].reset_index(drop=True)
print(f'  Splits: train={len(df_train):,}  val={len(df_val):,}  test={len(df_test):,}')
del df

def prepare_data(df_split):
    return {
        'states':            df_split[state_cols].values.astype(np.float32),
        'next_states':       df_split[next_state_cols].values.astype(np.float32),
        'actions':           df_split['a'].values.astype(np.int64),
        'rewards':           df_split['r'].values.astype(np.float32),
        'done':              df_split['done'].values.astype(np.float32),
        'phase':             df_split['phase'].values.astype(np.int64),
        'next_is_discharge': df_split['next_is_discharge'].values.astype(np.int64),
        'next_a_physician':  df_split['next_a_physician'].values.astype(np.int64),
    }

train_data = prepare_data(df_train)
val_data   = prepare_data(df_val)
test_data  = prepare_data(df_test)
print('Data ready.')

In [ ]:
# ── Cell 6: GPU-optimised joint training functions ────────────────────────────
#
# Same GPU optimisations as step_11b:
#   - Full dataset pre-loaded to GPU tensors once
#   - torch.multinomial sampling (GPU)
#   - Vectorised priority update
#   - Losses stored on GPU, single CPU transfer at end
#
# New vs step_11b:
#   - Two Q-networks: main_drug / main_discharge (+ targets)
#   - Batch split by phase mask
#   - Cross-phase Bellman: last in-stay blocs bootstrap from discharge target
#
# BatchNorm note:
#   Main networks are set to .eval() during target computation (inside no_grad)
#   so BatchNorm uses running statistics -- safe for any sub-batch size.
#   They are restored to .train() immediately before the loss backward pass.
#   Loss computation also guards against single-sample phases (>= 2 check).
#

def _load_to_gpu(data, dev):
    """Pre-load all arrays to GPU once."""
    return {
        'states':    torch.FloatTensor(data['states']).to(dev),
        'next':      torch.FloatTensor(data['next_states']).to(dev),
        'actions':   torch.LongTensor(data['actions']).to(dev),
        'rewards':   torch.FloatTensor(data['rewards']).to(dev),
        'done':      torch.FloatTensor(data['done']).to(dev),
        'phase':     torch.LongTensor(data['phase']).to(dev),
        'nid':       torch.BoolTensor(data['next_is_discharge'].astype(bool)).to(dev),
        'next_a':    torch.LongTensor(data['next_a_physician']).to(dev),
        'n':         len(data['states']),
        'priorities': (torch.FloatTensor(
                           np.abs(data['rewards']) + PER_EPSILON
                       ) ** PER_ALPHA).to(dev),
    }


def _sample_batch(g, batch_size):
    probs   = g['priorities'] / g['priorities'].sum()
    idx     = torch.multinomial(probs, batch_size, replacement=True)
    iw      = (g['n'] * probs[idx]) ** (-BETA_START)
    iw      = (iw / iw.max()).detach()
    return idx, iw


def _compute_targets(g, idx, main_drug, tgt_drug, main_discharge, tgt_discharge,
                     use_sarsa=False):
    # NOTE: main_drug and main_discharge must be in .eval() mode when this is called
    rewards = g['rewards'][idx]
    dones   = g['done'][idx]
    phases  = g['phase'][idx]
    nid     = g['nid'][idx]
    ns      = g['next'][idx]
    n       = len(idx)

    nq = torch.zeros(n, device=device)

    mask_reg  = (phases == 0) & (~nid)
    mask_last = (phases == 0) & nid

    if mask_reg.any():
        ns_r = ns[mask_reg]
        if use_sarsa:
            na_r = g['next_a'][idx][mask_reg]
            nq[mask_reg] = tgt_drug(ns_r).gather(1, na_r.unsqueeze(1)).squeeze(1)
        else:
            na_r = main_drug(ns_r).argmax(dim=1)
            nq[mask_reg] = tgt_drug(ns_r).gather(1, na_r.unsqueeze(1)).squeeze(1)

    if mask_last.any():
        ns_l = ns[mask_last]
        if use_sarsa:
            na_l = g['next_a'][idx][mask_last]
            nq[mask_last] = tgt_discharge(ns_l).gather(1, na_l.unsqueeze(1)).squeeze(1)
        else:
            na_l = main_discharge(ns_l).argmax(dim=1)
            nq[mask_last] = tgt_discharge(ns_l).gather(1, na_l.unsqueeze(1)).squeeze(1)

    nq = nq.clamp(-REWARD_THRESHOLD, REWARD_THRESHOLD)
    return rewards + GAMMA * nq * (1 - dones)


def train_joint(g, mode='ddqn',
                num_steps=DQN_STEPS, save_dir=None, label='DDQN'):
    """Unified loop for joint DDQN and joint SARSA."""
    use_sarsa = (mode == 'sarsa')

    main_drug        = DuelingDQN(n_state, N_DRUG_ACTIONS,      HIDDEN).to(device)
    tgt_drug         = DuelingDQN(n_state, N_DRUG_ACTIONS,      HIDDEN).to(device)
    main_discharge   = DuelingDQN(n_state, N_DISCHARGE_ACTIONS, HIDDEN).to(device)
    tgt_discharge    = DuelingDQN(n_state, N_DISCHARGE_ACTIONS, HIDDEN).to(device)

    tgt_drug.load_state_dict(main_drug.state_dict());           tgt_drug.eval()
    tgt_discharge.load_state_dict(main_discharge.state_dict()); tgt_discharge.eval()

    opt_drug      = optim.Adam(main_drug.parameters(),      lr=LR)
    opt_discharge = optim.Adam(main_discharge.parameters(), lr=LR)

    print(f'{label}: {num_steps} steps | n_state={n_state} | '
          f'drug={N_DRUG_ACTIONS} | discharge={N_DISCHARGE_ACTIONS} | '
          f'batch={BATCH_SIZE} | device={device}')

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    losses_gpu = torch.zeros(num_steps, device=device)
    t0 = time.time()

    for step in range(num_steps):
        idx, iw = _sample_batch(g, BATCH_SIZE)

        states  = g['states'][idx]
        actions = g['actions'][idx]
        phases  = g['phase'][idx]

        # --- Target computation: eval mode so BatchNorm uses running stats ---
        # Safe for any sub-batch size, including single samples.
        main_drug.eval();  main_discharge.eval()
        with torch.no_grad():
            targets = _compute_targets(
                g, idx, main_drug, tgt_drug, main_discharge, tgt_discharge,
                use_sarsa=use_sarsa,
            )

        # --- Loss computation: train mode for BatchNorm batch stats ----------
        main_drug.train();  main_discharge.train()

        mask0 = phases == 0
        mask1 = phases == 1
        loss  = torch.tensor(0.0, device=device)
        td_all = torch.zeros(BATCH_SIZE, device=device)

        # BatchNorm1d requires >= 2 samples; skip phase if too few in this batch
        if mask0.sum() >= 2:
            q0        = main_drug(states[mask0])
            q_taken0  = q0.gather(1, actions[mask0].unsqueeze(1)).squeeze(1)
            td0       = targets[mask0] - q_taken0
            loss0     = (iw[mask0] * td0.pow(2)).mean()
            reg0      = torch.clamp(q0.abs() - REWARD_THRESHOLD, min=0).sum()
            loss      = loss + loss0 + REG_LAMBDA * reg0 / BATCH_SIZE
            td_all[mask0] = td0.detach()

        if mask1.sum() >= 2:
            q1        = main_discharge(states[mask1])
            a1        = actions[mask1].clamp(0, N_DISCHARGE_ACTIONS - 1)
            q_taken1  = q1.gather(1, a1.unsqueeze(1)).squeeze(1)
            td1       = targets[mask1] - q_taken1
            loss1     = (iw[mask1] * td1.pow(2)).mean()
            reg1      = torch.clamp(q1.abs() - REWARD_THRESHOLD, min=0).sum()
            loss      = loss + loss1 + REG_LAMBDA * reg1 / BATCH_SIZE
            td_all[mask1] = td1.detach()

        opt_drug.zero_grad()
        opt_discharge.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(main_drug.parameters(),      10.0)
        nn.utils.clip_grad_norm_(main_discharge.parameters(), 10.0)
        opt_drug.step()
        opt_discharge.step()

        with torch.no_grad():
            g['priorities'][idx] = (td_all.abs() + PER_EPSILON) ** PER_ALPHA

        for pm, pt in zip(main_drug.parameters(),      tgt_drug.parameters()):
            pt.data.copy_(TAU * pm.data + (1 - TAU) * pt.data)
        for pm, pt in zip(main_discharge.parameters(), tgt_discharge.parameters()):
            pt.data.copy_(TAU * pm.data + (1 - TAU) * pt.data)

        losses_gpu[step] = loss.detach()

        if (step + 1) % LOG_EVERY == 0:
            elapsed   = time.time() - t0
            steps_sec = (step + 1) / elapsed
            eta       = (num_steps - step - 1) / steps_sec
            avg_loss  = losses_gpu[step+1-LOG_EVERY:step+1].mean().item()
            pct       = 100 * (step + 1) / num_steps
            print(f'  {label} {step+1}/{num_steps} ({pct:.0f}%) | '
                  f'loss={avg_loss:.4f} | {steps_sec:.1f} steps/s | '
                  f'elapsed {elapsed:.0f}s | ETA {eta:.0f}s')

        if save_dir and CHECKPOINT_EVERY and (step + 1) % CHECKPOINT_EVERY == 0:
            torch.save(main_drug.state_dict(),
                       os.path.join(save_dir, f'checkpoint_drug_{step+1}.pt'))
            torch.save(main_discharge.state_dict(),
                       os.path.join(save_dir, f'checkpoint_discharge_{step+1}.pt'))
            print(f'  Checkpoint saved at step {step+1}')

    # Final Q-values
    main_drug.eval();  main_discharge.eval()
    n_total = g['n']
    dq_list, dischq_list = [], []
    with torch.no_grad():
        for i in range(0, n_total, 4096):
            s_batch = g['states'][i:i+4096]
            dq_list.append(   main_drug(s_batch).cpu().numpy())
            dischq_list.append(main_discharge(s_batch).cpu().numpy())
    drug_q  = np.concatenate(dq_list,    axis=0)
    disch_q = np.concatenate(dischq_list, axis=0)

    losses = losses_gpu.cpu().numpy().tolist()
    print(f'{label} complete. Drug mean Q={drug_q.mean():.4f} | '
          f'Discharge mean Q={disch_q.mean():.4f}')

    if save_dir:
        prefix = 'dqn' if not use_sarsa else 'sarsa'
        phys   = '' if not use_sarsa else 'phys_'
        torch.save(main_drug.state_dict(),
                   os.path.join(save_dir, f'{prefix}_drug_model.pt'))
        torch.save(main_discharge.state_dict(),
                   os.path.join(save_dir, f'{prefix}_discharge_model.pt'))
        for name, arr in [
            (f'{phys}drug_actions.pkl',       drug_q.argmax(axis=1)),
            (f'{phys}drug_q_values.pkl',      drug_q),
            (f'{phys}discharge_actions.pkl',  disch_q.argmax(axis=1)),
            (f'{phys}discharge_q_values.pkl', disch_q),
            (f'{prefix}_losses.pkl',          losses),
        ]:
            with open(os.path.join(save_dir, name), 'wb') as f:
                pickle.dump(arr, f)
        print(f'  Saved to {save_dir}')

    return main_drug, main_discharge, drug_q, disch_q


def compute_split_outputs(drug_model, discharge_model, data_split, split_name, save_dir, prefix):
    """Compute and save Q-values for a val/test split."""
    st = torch.FloatTensor(data_split['states']).to(device)
    with torch.no_grad():
        dq_list, dischq_list = [], []
        for i in range(0, len(st), 4096):
            dq_list.append(   drug_model(st[i:i+4096]).cpu().numpy())
            dischq_list.append(discharge_model(st[i:i+4096]).cpu().numpy())
    drug_q  = np.concatenate(dq_list,    axis=0)
    disch_q = np.concatenate(dischq_list, axis=0)

    phys = '' if 'dqn' in prefix else 'phys_'
    p    = 'dqn' if 'dqn' in prefix else 'phys'
    for name, arr in [
        (f'{phys}drug_actions_{split_name}.pkl',       drug_q.argmax(axis=1)),
        (f'{phys}drug_q_{split_name}.pkl',             drug_q),
        (f'{phys}discharge_actions_{split_name}.pkl',  disch_q.argmax(axis=1)),
        (f'{phys}discharge_q_{split_name}.pkl',        disch_q),
    ]:
        with open(os.path.join(save_dir, name), 'wb') as f:
            pickle.dump(arr, f)

    phase1 = data_split['phase'] == 1
    da_p1  = disch_q[phase1].argmax(axis=1) if phase1.any() else np.array([])
    dist   = np.bincount(da_p1, minlength=N_DISCHARGE_ACTIONS).tolist() if len(da_p1) else []
    print(f'  {p} {split_name}: drug mean Q={drug_q.mean():.4f} | '
          f'discharge dist (phase-1): {dist}')


print('GPU-optimised joint training functions defined')

In [ ]:
# ── Cell 7: SMOKE TEST ────────────────────────────────────────────────────────
# 500 steps each. All assertions must pass before proceeding.

SMOKE_DIR   = '/tmp/smoke_tier2_discharge'
SMOKE_STEPS = 500
SMOKE_BATCH_ORIG = BATCH_SIZE
BATCH_SIZE  = 64    # small for smoke

print('=' * 60)
print('SMOKE TEST -- 500 steps joint DDQN + 500 steps joint SARSA')
print('=' * 60)

g_smoke = _load_to_gpu(train_data, device)

sm_drug, sm_disch, sm_dq, sm_dischq = train_joint(
    g_smoke, mode='ddqn', num_steps=SMOKE_STEPS,
    save_dir=os.path.join(SMOKE_DIR, 'ddqn'), label='Smoke-DDQN')

g_smoke2 = _load_to_gpu(train_data, device)
sm_sdrug, sm_sdisch, sm_sdq, sm_sdischq = train_joint(
    g_smoke2, mode='sarsa', num_steps=SMOKE_STEPS,
    save_dir=os.path.join(SMOKE_DIR, 'sarsa'), label='Smoke-SARSA')

BATCH_SIZE = SMOKE_BATCH_ORIG   # restore

errors = []
n_train = len(train_data['states'])

if n_state != 5:
    errors.append(f'n_state={n_state}, expected 5')
if sm_dq.shape != (n_train, N_DRUG_ACTIONS):
    errors.append(f'Drug Q shape {sm_dq.shape}, expected ({n_train}, {N_DRUG_ACTIONS})')
if sm_dischq.shape != (n_train, N_DISCHARGE_ACTIONS):
    errors.append(f'Discharge Q shape {sm_dischq.shape}, expected ({n_train}, {N_DISCHARGE_ACTIONS})')
if not np.isfinite(sm_dq).all():
    errors.append('Drug Q contains NaN/Inf')
if not np.isfinite(sm_dischq).all():
    errors.append('Discharge Q contains NaN/Inf')
if sm_dq.argmax(axis=1).min() < 0 or sm_dq.argmax(axis=1).max() >= N_DRUG_ACTIONS:
    errors.append('Drug actions out of range')
if sm_dischq.argmax(axis=1).min() < 0 or sm_dischq.argmax(axis=1).max() >= N_DISCHARGE_ACTIONS:
    errors.append('Discharge actions out of range')

# Check phase-1 rows have discharge actions in valid range
p1_mask = train_data['phase'] == 1
if p1_mask.any():
    p1_actions = sm_dischq[p1_mask].argmax(axis=1)
    if p1_actions.max() >= N_DISCHARGE_ACTIONS:
        errors.append('Phase-1 discharge actions out of range')

for fname in ['dqn_drug_model.pt', 'dqn_discharge_model.pt', 'dqn_losses.pkl']:
    if not os.path.exists(os.path.join(SMOKE_DIR, 'ddqn', fname)):
        errors.append(f'Missing file: {fname}')

print()
if errors:
    for e in errors:
        print(f'  FAIL: {e}')
    raise RuntimeError('Smoke test FAILED -- do not proceed to full training')
else:
    print(f'  n_state:              {n_state} (expect 5)')
    print(f'  n_drug_actions:       {N_DRUG_ACTIONS} (expect 16)')
    print(f'  n_discharge_actions:  {N_DISCHARGE_ACTIONS} (expect 3)')
    print(f'  Drug Q shape:         {sm_dq.shape}')
    print(f'  Discharge Q shape:    {sm_dischq.shape}')
    print(f'  Drug unique actions:  {sorted(np.unique(sm_dq.argmax(axis=1)).tolist())}')
    if p1_mask.any():
        print(f'  Discharge dist(p1):   {np.bincount(sm_dischq[p1_mask].argmax(axis=1), minlength=3).tolist()}')
    print(f'  Phase-1 rows:         {p1_mask.sum()} (expect ~{len(train_data["states"]) // 25} approx)')
    print()
    print('SMOKE TEST PASSED -- safe to proceed to full training')

In [ ]:
# ── Cell 8: Train Joint DDQN ──────────────────────────────────────────────────
g_train = _load_to_gpu(train_data, device)

dqn_drug, dqn_discharge, dqn_drug_q, dqn_discharge_q = train_joint(
    g_train, mode='ddqn', num_steps=DQN_STEPS,
    save_dir=DDQN_DIR, label='Joint-DDQN')

for split_name, split_data in [('val', val_data), ('test', test_data)]:
    compute_split_outputs(dqn_drug, dqn_discharge, split_data,
                          split_name, DDQN_DIR, 'dqn')

In [ ]:
# ── Cell 9: Train Joint SARSA physician baseline ──────────────────────────────
g_train2 = _load_to_gpu(train_data, device)

sarsa_drug, sarsa_discharge, sarsa_drug_q, sarsa_discharge_q = train_joint(
    g_train2, mode='sarsa', num_steps=SARSA_STEPS,
    save_dir=SARSA_DIR, label='Joint-SARSA')

for split_name, split_data in [('val', val_data), ('test', test_data)]:
    compute_split_outputs(sarsa_drug, sarsa_discharge, split_data,
                          split_name, SARSA_DIR, 'phys')

In [ ]:
# ── Cell 10: Verify outputs ───────────────────────────────────────────────────
print('Files saved to Drive:')
for root, dirs, files in os.walk(MODEL_DIR):
    for fname in sorted(files):
        path    = os.path.join(root, fname)
        size_kb = os.path.getsize(path) / 1024
        rel     = os.path.relpath(path, MODEL_DIR)
        print(f'  {rel:60s}  {size_kb:8.1f} KB')

# Quick discharge policy summary
print()
print('Discharge policy summary (train, phase-1 rows):')
p1 = train_data['phase'] == 1
if p1.any():
    obs_actions  = train_data['actions'][p1]
    dqn_actions  = dqn_discharge_q[p1].argmax(axis=1)
    sarsa_actions = sarsa_discharge_q[p1].argmax(axis=1)
    for label, acts in [('Observed physician', obs_actions),
                        ('DDQN policy',        dqn_actions),
                        ('SARSA physician',    sarsa_actions)]:
        dist = np.bincount(acts, minlength=N_DISCHARGE_ACTIONS)
        pcts = 100 * dist / dist.sum()
        row  = '  '.join(f'{DISCHARGE_LABELS[i]}={pcts[i]:.1f}%' for i in range(N_DISCHARGE_ACTIONS))
        print(f'  {label:<22s}: {row}')

print()
print('Done. Download models/tier2_discharge/ from Drive and place in:')
print('  CareAI/models/icu_readmit/tier2_discharge/')